In [59]:
import requests
from dataclasses import fields, dataclass
from datetime import datetime, timedelta
from pyiceberg.catalog import load_catalog
from pyiceberg.schema import (
    Schema, 
    NestedField, 
    FloatType, 
    IntegerType,
    LongType,
    TimestampType, 
    StringType
)
import polars as pl
import duckdb
import json


today = datetime.now()
first_of_month = datetime(today.year, today.month, 1)
today_str = today.strftime("%Y-%m-%d")
fom_str = first_of_month.strftime("%Y-%m-%d")

cat = load_catalog("dev")

TYPE_MAP = {
    str: StringType(),
    int: LongType(),
    float: FloatType(),
    datetime: TimestampType(),
}

def schema_from_dataclass(cls):
    iceberg_fields = []
    for i, f in enumerate(fields(cls), start=1):
        iceberg_fields.append(
            NestedField(
                i,
                f.name,
                TYPE_MAP[f.type],
                is_optional=True
            )
        )
    return Schema(*iceberg_fields)

def daterange(start_date, end_date, step=timedelta(days=1)):
    """
    A generator function that yields dates between start_date and end_date.
    The end_date is not included in the range.
    """
    current_date = start_date
    while current_date < end_date:
        yield current_date
        current_date += step

print(f'duckdb.version: {duckdb.__version__}')

duckdb.version: 1.5.0.dev334


In [ ]:

try:
    con.execute("""
        DETACH dev;
    """)
    
    if con is not None:
        con.close()
except:
    pass
con = duckdb.connect('duck.db')

con.sql("""
    INSTALL httpfs;
    LOAD httpfs;
    CREATE OR REPLACE SECRET minio (
        TYPE S3,
        KEY_ID 'minioadmin',
        SECRET 'minioadmin',
        ENDPOINT 'minio:9000', -- Remove http:// prefix here
        REGION 'us-east-1',
        USE_SSL false,
        URL_STYLE 'path'
    );
""")

con.sql("""
    INSTALL iceberg;
    LOAD iceberg;
    
    ATTACH 'dev' (
        TYPE iceberg
        , AUTHORIZATION_TYPE none
        , ENDPOINT 'http://rest:8181'
    );

    --show schemas 
""")
con.sql("show tables from dev.silver").pl()


In [ ]:
@dataclass
class NhlApiCall:
    requested_at: datetime
    endpoint: str
    status: int
    response: str

schema = schema_from_dataclass(NhlApiCall)
cat.create_table_if_not_exists("bronze.nhl_api_calls", schema=schema)

table = cat.load_table("bronze.nhl_api_calls")
df = pl.from_arrow(table.scan().to_arrow())
endpoint = "https://api-web.nhle.com/v1/schedule/{day}"
for dt in [datetime(today.year, today.month, i+1) for i in range(today.day)]:
    dt_str = dt.strftime("%Y-%m-%d")
    parsed_endpoint = endpoint.format(day=dt_str)
    if dt in (df.filter(
        (pl.col("status") == 200)
        & (pl.col("endpoint") == parsed_endpoint)
    )):
        continue
        
    response = requests.get(parsed_endpoint)
    df.extend(pl.DataFrame({
        "requested_at": datetime.now(),
        "endpoint": parsed_endpoint,
        "status": response.status_code,
        "response": response.text
    }))

df.write_iceberg("bronze.nhl_api_calls", catalog=catalog, mode="overwrite")

In [71]:
con.sql("""
    CREATE TABLE IF NOT EXISTS dev.bronze.request_queue (
        request_id UUID,
        endpoint TEXT,
        status TEXT,
        created_at TIMESTAMP
    )
""")

def add_request_to_queue(endpoint):
    con.execute("""
        INSERT INTO dev.bronze.request_queue (request_id, endpoint, status, created_at)
        VALUES (uuid(), ?, 'pending', NOW())
    """, [endpoint])

def process_request_queue():
    to_update = []
    df = con.sql("""
        SELECT 
            request_id, 
            endpoint
        FROM dev.bronze.request_queue WHERE status = 'pending'
        ORDER BY created_at 
        LIMIT 10
    """).pl()
    api_calls = cat.load_table('bronze.nhl_api_calls')
    schema = api_calls.schema().as_arrow()
    rows = []
    for request_id, endpoint in df.rows():
        response = requests.get(endpoint, timeout=30)
        to_update.append(request_id)
        rows.append({
            "requested_at": datetime.now(),
            "endpoint": endpoint,
            "status": response.status_code,
            "response": response.text    
        })
        
    if df.is_empty():
        return

    df_rows = pl.DataFrame(rows)
    df_rows.write_iceberg(api_calls, mode="append")
    con.execute("""UPDATE dev.bronze.request_queue 
            SET status='done' 
            WHERE request_id IN 
            (
            SELECT CAST(x AS UUID)
            FROM UNNEST(?) t(x)
            )""", [to_update])
    print(f"{len(to_update)} requests atualizados!")
    

In [ ]:

schema = schema_from_dataclass(NhlApiCall)
cat.create_table_if_not_exists("bronze.nhl_api_calls", schema=schema)
table = cat.load_table("bronze.nhl_api_calls")
print(table.schema)
df.write_iceberg(table, 
                 mode="overwrite")
df

In [ ]:
cat = load_catalog("dev")

In [ ]:
tbl = cat.load_table("bronze.nhl_api_calls")
df = pl.from_arrow(tbl.scan().to_arrow())
df.head(10)

In [ ]:
# Bronze to silver
table = cat.load_table('bronze.nhl_api_calls')
df = pl.scan_iceberg(table, storage_options=cat.properties)
# Requests para 
df = (
    df
    .filter(pl.col('status') == 200)
    .filter(pl.col('endpoint').str.contains('/v1/schedule/'))
    # .filter(pl.col('response').str.contains('data'))
    .group_by("endpoint")
    .agg(
        pl.all().sort_by("requested_at").last()
    )
)

df_e = df.collect()

df_games = (
    df_e
    .with_columns(
        pl.col("response").alias('json_response'),
        pl.col("endpoint").str.extract(r"/schedule/(\d{4}-\d{2}-\d{2})").str.to_date().alias("endpoint_date")
    ).map_columns('json_response', lambda s: s.str.json_decode())
)

df_games = df_games.lazy()

df_games = (
    df_games
    .with_columns(pl.col('json_response').struct.field('gameWeek'))
    .explode('gameWeek')
    .unnest('gameWeek')
    .explode('games')
    .unnest('games')
    .with_columns(
        pl.col('venue').struct.rename_fields(['venue'])
    )
    .unnest('venue')
)

team_fields = [f.name for f in df_games.limit(1).collect().schema["homeTeam"].fields]
new_field_names_home = [f"home_{s}" for s in team_fields]
new_field_names_away = [f"away_{s}" for s in team_fields]

df_games = (
    df_games
    .with_columns(
        pl.col('homeTeam').struct.rename_fields([f'home_{f}' for f in team_fields]),
        pl.col('awayTeam').struct.rename_fields([f'away_{f}' for f in team_fields])
    )
    .unnest(['homeTeam', 'awayTeam'])
    .with_columns([
        pl.col('home_commonName').struct.field('default').alias('home_commom_name'),
        pl.col('away_commonName').struct.field('default').alias('away_commom_name'),
    ])
    .rename({"id": "game_id"})
    .filter(
        (pl.col("game_id").is_not_null())
        &(pl.col("gameType").is_in([1,2,3]))
    )
    .with_columns([
        pl.col("date").str.to_date(),
        pl.col("startTimeUTC").str.to_datetime("%Y-%m-%dT%H:%M:%SZ", time_zone="UTC")
    ])
    .rename({
        "startTimeUTC": "start_time_utc", 
        "gameState": "game_state", 
        "gameScheduleState": "game_schedule_state"
    })
    .drop("response", 
          "json_response", "status", 
          "numberOfGames", 'datePromo', 
          'dayAbbrev', 'season', 'gameType',
          'neutralSite', 'easternUTCOffset',
          'venueUTCOffset', 'venueTimezone', 
          'away_commonName', 'home_commonName'
         )
    .sort(["endpoint_date", "requested_at"])
)

silver_df = (
   df_games.select("date", "game_id", "venue",
                    "start_time_utc", "game_state", "game_schedule_state", 
                    "away_id", "away_abbrev", "away_commom_name",
                    "home_id", "home_abbrev", "home_commom_name",
                    "endpoint", "endpoint_date", "requested_at"
                   ).group_by("game_id").agg(pl.all().last()) 
    .collect()
)

cat.create_table_if_not_exists("silver.nhl_games", schema=silver_df.to_arrow().schema)
table = cat.load_table("silver.nhl_games")
silver_df.write_iceberg(table, mode="overwrite")

In [87]:
#play by play: 
from datetime import timedelta
endpoint = "https://api-web.nhle.com/v1/gamecenter/{}/play-by-play"

prev30 = datetime.now() - timedelta(days=90)
table = cat.load_table('silver.nhl_games')
requests_list = (
    pl.scan_iceberg(
        cat.load_table('bronze.nhl_api_calls'),
        storage_options=cat.properties
    )
    .filter(
        (pl.col("status") == 200) & 
        (pl.col("endpoint").str.contains("play-by-play")) &
        (pl.col("requested_at") >= prev30)
    )
    .select("endpoint")
    .collect()
)["endpoint"].to_list()
df = pl.scan_iceberg(table, storage_options=cat.properties)
df = (
    df.select("game_id").unique()
    .with_columns(
        pl.format(endpoint, pl.col("game_id")).alias("endpoint")
    )
    .filter(~pl.col("endpoint").is_in(requests_list))
).collect()
todo = df["endpoint"].to_list()

for endpoint in todo:
    add_request_to_queue(endpoint)

In [55]:
con.execute("""
    CREATE TABLE IF NOT EXISTS dev.bronze.ingestion (
        batch_id VARCHAR,
        event_time TIMESTAMP,
        ingestion_name VARCHAR,
        source_table VARCHAR,
        target_table VARCHAR,
        final_status VARCHAR, -- FAIL, SUCCESS, NULL
        event_type VARCHAR -- STARTED, FINISHED
    ); -- PARTITION BY (date(ingested_at), ingestion_name);

    CREATE TABLE IF NOT EXISTS dev.bronze.ingestion_messages (
        ingestion_id VARCHAR,
        time_stamp TIMESTAMP,
        message VARCHAR,
        level VARCHAR
    );

    CREATE TABLE IF NOT EXISTS dev.bronze.ingestion_params (
        ingestion_id VARCHAR,
        param_name VARCHAR,
        param_value VARCHAR
    )
""")

In [96]:
q = (
    pl.from_arrow(
        cat.load_table("bronze.nhl_api_calls").scan().to_arrow()
    )
    .filter(
        (pl.col("endpoint").str.contains("play-by-play"))
        & (pl.col("status") == 200)
    )
    .sort("requested_at")
    .group_by("endpoint").agg(pl.all().last())
)["response", "endpoint"]

query = """
    SELECT
        api.endpoint,
        params.ingestion_id AS batch_id,
        COUNT(plays.game_id) AS eventos,
        COUNT(DISTINCT plays.game_id) AS jogos
    FROM dev.bronze.nhl_api_calls api
    
    LEFT JOIN dev.bronze.ingestion_params params
        ON params.param_value = api.endpoint
        AND params.param_name = 'endpoint'
    
    LEFT JOIN dev.silver.nhl_play_by_play plays
        ON plays.batch_id = params.ingestion_id

    LEFT JOIN dev.silver.nhl_games schedule
        ON plays.game_id = schedule.game_id
    
    WHERE api.endpoint LIKE '%play-by-play%'
    AND api.status = 200
    AND schedule.date < TODAY()
    
    GROUP BY api.endpoint, params.ingestion_id
    ORDER BY batch_id;
"""

df_ingestoes = (
    con.sql(query).pl()
    .filter(pl.col('jogos') == 0)
    .select("endpoint")
)
print(f"Registros a ingestar: {df_ingestoes.height}")
q = q.filter(pl.col("endpoint").is_in(df_ingestoes["endpoint"].to_list()))

import uuid

def ingest_status():
    ingestion_name = "nhl_play_by_play"
    source_table = "bronze.nhl_api_calls"
    target_table = "silver.nhl_play_by_play"
    def wrapper(batch_id, final_status, event_type):
        con.execute("""
                INSERT INTO dev.bronze.ingestion (
                    batch_id,
                    event_time,
                    ingestion_name,
                    source_table,
                    target_table,
                    final_status,
                    event_type
                ) VALUES (?, NOW(), ?, ?, ?, ?, ?)
            """, [
                batch_id,
                ingestion_name,
                source_table,
                target_table,
                final_status,
                event_type
            ])
        return
    return wrapper
    
def save_params(batch_id, param_dict):
    rows = []
    for key, value in param_dict.items():
        if isinstance(value, (dict, list)):
            value = json.dumps(value)
        else:
            value = str(value)
        
        rows.append((batch_id, key, value))
    con.executemany("""
        INSERT INTO dev.bronze.ingestion_params
        (ingestion_id, param_name, param_value)
        VALUES (?, ?, ?)
    """, rows)

event_logger = ingest_status()
table = cat.load_table("silver.nhl_play_by_play")
for resp, endpoint in q.rows():
    payload = json.loads(resp)
    batch_id = str(uuid.uuid4())
    rows = []
    event_logger(batch_id, None, "STARTED")
    save_params(batch_id, {"endpoint": endpoint})
    print(f"⏳ Processando dados para {endpoint}")
    try:
        for play in payload["plays"]:
            row = {
                "batch_id": batch_id,
                "game_id": payload["id"],
                "event_id": play["eventId"],
                "period": play["periodDescriptor"]["number"],
                "time_period": play["timeInPeriod"],
                "situation_code": play["situationCode"],
                "home_team_defending_side": play["homeTeamDefendingSide"],
                "type_code": play["typeCode"],
                "type_desc": play["typeDescKey"],
                "sort_order": play["sortOrder"],
            }
            if "details" in play:
                row.update({
                    "x_coord": play["details"].get("xCoord"),
                    "y_coord": play["details"].get("yCoord"),
                    "zone_code": play["details"].get("zoneCode"),
                    "event_owner_team_id": play["details"].get("eventOwnerTeamId")
                })
            rows.append(row)
        if len(rows) == 0:
            print(f"Adicionando {endpoint} à queue")
            add_request_to_queue(endpoint)
            raise ValueError("Faltando campo 'plays' no payload")
        pl.DataFrame(rows).write_iceberg(table, mode="append")
        event_logger(batch_id, "SUCCESS", "FINISHED")
        print(f"Registrados {len(rows)} eventos")
    except Exception as e:
        event_logger(batch_id, "FAIL", "FINISHED")
        con.execute("""INSERT INTO dev.bronze.ingestion_messages 
                        (ingestion_id, time_stamp, message, level) 
                        VALUES (?, NOW(), ?, 'ERROR')""", [batch_id, str(e)])
print("Finalizado")

Registros a ingestar: 0
Finalizado


In [91]:
con.sql("""
    CREATE TABLE IF NOT EXISTS dev.silver.nhl_play_by_play 
    (
        batch_id VARCHAR,
        game_id LONG,
        event_id LONG,
        period LONG,
        time_period VARCHAR,
        situation_code VARCHAR,
        home_team_defending_side VARCHAR,
        type_code LONG
        , type_desc VARCHAR
        , sort_order LONG
        , x_coord LONG
        , y_coord LONG
        , zone_code VARCHAR
        , event_owner_team_id LONG
    )
    """)


In [90]:
con.sql("drop table dev.silver.nhl_play_by_play")

In [97]:
con.sql("select * from  dev.bronze.ingestion WHERE event_type='FINISHED' AND target_table = 'silver.nhl_play_by_play'").pl()
con.sql("""
    SELECT batch_id, game_id, count() as 'eventos'
    FROM dev.silver.nhl_play_by_play 
    GROUP BY game_id, batch_id
""").pl()
con.sql(query)

┌─────────────────────────────────────────────────────────────┬──────────────────────────────────────┬─────────┬───────┐
│                          endpoint                           │               batch_id               │ eventos │ jogos │
│                           varchar                           │               varchar                │  int64  │ int64 │
├─────────────────────────────────────────────────────────────┼──────────────────────────────────────┼─────────┼───────┤
│ https://api-web.nhle.com/v1/gamecenter/2025020949/play-by-… │ 03ab553a-0e0c-4861-b692-264a2b92994c │     576 │     1 │
│ https://api-web.nhle.com/v1/gamecenter/2025020878/play-by-… │ 0996e8fe-c39f-49a1-b0a3-08aa558bf645 │     640 │     1 │
│ https://api-web.nhle.com/v1/gamecenter/2025020911/play-by-… │ 119548c3-ec52-45ae-89a4-25265530befb │     586 │     1 │
│ https://api-web.nhle.com/v1/gamecenter/2025020942/play-by-… │ 11ba2a65-d835-4693-870b-eedde73d7e8f │     640 │     1 │
│ https://api-web.nhle.com/v1/ga

In [98]:
con.sql("""
    select m.message, p.param_value     
    from dev.bronze.ingestion_messages m
    left join dev.bronze.ingestion i on (i.batch_id = m.ingestion_id and event_type='FINISHED' AND i.final_status='FAIL')
    left join dev.bronze.ingestion_params p on (p.ingestion_id = i.batch_id and p.param_name = 'endpoint')
    LIMIT 10;

    SELECT 
        api.requested_at
        , params.ingestion_id
        , params.param_value as 'endpoint'
        , ing.final_status
        , ing.event_type
        , plays.eventos
        , plays.game_id
    FROM dev.bronze.ingestion ing
    JOIN dev.bronze.ingestion_params params 
      ON ing.batch_id = params.ingestion_id
    LEFT JOIN dev.bronze.nhl_api_calls api 
      ON params.param_value = api.endpoint
    LEFT JOIN (
        SELECT batch_id, game_id, count() as 'eventos' 
        FROM dev.silver.nhl_play_by_play 
        GROUP BY game_id, batch_id
    ) plays 
      ON plays.batch_id = ing.batch_id
   
    WHERE 
        api.endpoint LIKE '%play-by-play'
        AND status = 200
        AND ing.event_type = 'FINISHED'
        AND ing.ingestion_name = 'nhl_play_by_play'
        --AND plays.game_id IS NULL
    ORDER BY game_id
""").pl()

requested_at,ingestion_id,endpoint,final_status,event_type,eventos,game_id
datetime[μs],str,str,str,str,i64,i64
2026-03-05 20:09:28.470168,"""1bc09e71-20de-4fa2-9ce5-cf8ebd…","""https://api-web.nhle.com/v1/ga…","""SUCCESS""","""FINISHED""",376,2025020873
2026-03-04 01:56:22.463916,"""1bc09e71-20de-4fa2-9ce5-cf8ebd…","""https://api-web.nhle.com/v1/ga…","""SUCCESS""","""FINISHED""",376,2025020873
2026-03-04 01:56:27.994929,"""4f71465a-3912-417c-bbb8-84d94e…","""https://api-web.nhle.com/v1/ga…","""SUCCESS""","""FINISHED""",329,2025020874
2026-03-05 20:07:28.772146,"""4f71465a-3912-417c-bbb8-84d94e…","""https://api-web.nhle.com/v1/ga…","""SUCCESS""","""FINISHED""",329,2025020874
2026-03-05 20:09:08.677628,"""2b043836-3883-4291-85aa-dfe551…","""https://api-web.nhle.com/v1/ga…","""SUCCESS""","""FINISHED""",325,2025020875
…,…,…,…,…,…,…
2026-03-05 20:09:45.231823,"""b1f8c22e-ea3f-415b-b227-a20d72…","""https://api-web.nhle.com/v1/ga…","""FAIL""","""FINISHED""",null,null
2026-03-04 01:56:14.687539,"""c35e774f-62ab-45b6-a482-8b0559…","""https://api-web.nhle.com/v1/ga…","""SUCCESS""","""FINISHED""",null,null
2026-03-05 20:08:23.168954,"""c35e774f-62ab-45b6-a482-8b0559…","""https://api-web.nhle.com/v1/ga…","""SUCCESS""","""FINISHED""",null,null


In [7]:
%%duck
SELECT 
        api.requested_at
        , params.ingestion_id
        , params.param_value as 'endpoint'
        , ing.final_status
        , ing.event_type
        , plays.eventos
        , plays.game_id
    FROM dev.bronze.ingestion ing
    JOIN dev.bronze.ingestion_params params 
      ON ing.batch_id = params.ingestion_id
    LEFT JOIN dev.bronze.nhl_api_calls api 
      ON params.param_value = api.endpoint
    LEFT JOIN (
        SELECT batch_id, game_id, count() as 'eventos' 
        FROM dev.silver.nhl_play_by_play 
        GROUP BY game_id, batch_id
    ) plays 
      ON plays.batch_id = ing.batch_id
   
    WHERE 
        api.endpoint LIKE '%play-by-play'
        AND status = 200
        AND ing.event_type = 'FINISHED'
        AND ing.ingestion_name = 'nhl_play_by_play'
        --AND plays.game_id IS NULL
    ORDER BY game_id

requested_at,ingestion_id,endpoint,final_status,event_type,eventos,game_id
datetime[μs],str,str,str,str,i64,i64
2026-03-04 01:56:22.463916,"""1bc09e71-20de-4fa2-9ce5-cf8ebd…","""https://api-web.nhle.com/v1/ga…","""SUCCESS""","""FINISHED""",376,2025020873
2026-03-04 01:56:27.994929,"""4f71465a-3912-417c-bbb8-84d94e…","""https://api-web.nhle.com/v1/ga…","""SUCCESS""","""FINISHED""",329,2025020874
2026-03-04 01:56:34.691169,"""2b043836-3883-4291-85aa-dfe551…","""https://api-web.nhle.com/v1/ga…","""SUCCESS""","""FINISHED""",325,2025020875
2026-03-04 01:56:20.330546,"""4e9e85ee-4a77-4712-8484-673f09…","""https://api-web.nhle.com/v1/ga…","""SUCCESS""","""FINISHED""",344,2025020876
2026-03-04 01:56:19.400916,"""fdc7d4be-e240-4200-92a0-276288…","""https://api-web.nhle.com/v1/ga…","""SUCCESS""","""FINISHED""",303,2025020877
…,…,…,…,…,…,…
2026-03-04 01:56:43.902281,"""577b418e-4b2b-4cd1-9035-a6c1c7…","""https://api-web.nhle.com/v1/ga…","""SUCCESS""","""FINISHED""",null,null
2026-03-04 01:56:31.056208,"""ed153e89-0365-4b94-a8a3-705daf…","""https://api-web.nhle.com/v1/ga…","""SUCCESS""","""FINISHED""",null,null
2026-03-04 01:56:32.530258,"""665b7c83-8343-41de-8c2d-44e7c3…","""https://api-web.nhle.com/v1/ga…","""SUCCESS""","""FINISHED""",null,null


In [18]:
%%duck
    SELECT 
        api.endpoint, 
        ingestions.count,
        plays.*
    FROM dev.bronze.nhl_api_calls api
    LEFT JOIN (
        SELECT 
            params.param_value as 'endpoint', 
            count() as count 
        FROM dev.bronze.ingestion_params params
        WHERE params.param_name = 'endpoint'
        GROUP BY params.param_value
    ) ingestions ON api.endpoint = ingestions.endpoint
    LEFT JOIN (
        SELECT 
            batch_id, 
            game_id,
            params.param_value as 'endpoint', 
            count() as 'eventos' 
        FROM dev.silver.nhl_play_by_play plays
            LEFT JOIN dev.bronze.ingestion_params params 
                ON params.ingestion_id = plays.batch_id
                    AND params.param_name = 'endpoint'
        GROUP BY game_id, batch_id, params.param_value
    ) plays 
        ON (plays.endpoint = api.endpoint)
    WHERE api.endpoint LIKE '%play-by-play%'
        AND api.status = 200

endpoint,count,batch_id,game_id,endpoint_1,eventos
str,i64,str,i64,str,i64
"""https://api-web.nhle.com/v1/ga…",1,"""2c873363-86e5-4b25-905e-e2525d…",2025020962,"""https://api-web.nhle.com/v1/ga…",212
"""https://api-web.nhle.com/v1/ga…",1,"""6d4f64f4-a3bd-4d4b-915e-e36bcf…",2025020900,"""https://api-web.nhle.com/v1/ga…",290
"""https://api-web.nhle.com/v1/ga…",1,"""ea8ee318-ad7f-4092-b25d-ddfc1e…",2025020896,"""https://api-web.nhle.com/v1/ga…",344
"""https://api-web.nhle.com/v1/ga…",1,"""2e819267-f131-43e0-bd6d-ebca67…",2025020951,"""https://api-web.nhle.com/v1/ga…",283
"""https://api-web.nhle.com/v1/ga…",1,"""754a5f02-bd75-4532-9f4e-b83335…",2025020889,"""https://api-web.nhle.com/v1/ga…",312
…,…,…,…,…,…
"""https://api-web.nhle.com/v1/ga…",1,null,null,null,null
"""https://api-web.nhle.com/v1/ga…",1,null,null,null,null
"""https://api-web.nhle.com/v1/ga…",1,null,null,null,null


In [62]:
%%duck
SELECT
    api.endpoint,
    params.ingestion_id AS batch_id,
    COUNT(plays.game_id) AS eventos,
    COUNT(DISTINCT plays.game_id) AS jogos
FROM dev.bronze.nhl_api_calls api

LEFT JOIN dev.bronze.ingestion_params params
    ON params.param_value = api.endpoint
    AND params.param_name = 'endpoint'

LEFT JOIN dev.silver.nhl_play_by_play plays
    ON plays.batch_id = params.ingestion_id

WHERE api.endpoint LIKE '%play-by-play%'
AND api.status = 200

GROUP BY api.endpoint, params.ingestion_id
ORDER BY batch_id;

endpoint,batch_id,eventos,jogos
str,str,i64,i64
"""https://api-web.nhle.com/v1/ga…","""00430716-84f6-41fd-a578-853ea7…",0,0
"""https://api-web.nhle.com/v1/ga…","""03ab553a-0e0c-4861-b692-264a2b…",288,1
"""https://api-web.nhle.com/v1/ga…","""06fe1164-c03c-42f1-82d4-08eede…",0,0
"""https://api-web.nhle.com/v1/ga…","""0996e8fe-c39f-49a1-b0a3-08aa55…",320,1
"""https://api-web.nhle.com/v1/ga…","""09a8c632-4e72-4df8-8ad8-3c2dab…",0,0
…,…,…,…
"""https://api-web.nhle.com/v1/ga…","""f771a7c0-495a-4b63-b5fc-b1559e…",310,1
"""https://api-web.nhle.com/v1/ga…","""f87bacad-5f4d-4d8e-bcf6-e68f63…",297,1
"""https://api-web.nhle.com/v1/ga…","""fa807ad4-7c1f-40d3-8b9e-8cb4b3…",0,0


In [67]:
endpoint = "https://api-web.nhle.com/v1/schedule/{day}"
for dt in daterange(datetime(2026,1,1), datetime.today()):
    dt_str = dt.strftime("%Y-%m-%d")
    parsed_endpoint = endpoint.format(day=dt_str)
    add_request_to_queue(parsed_endpoint)

# Processamento de requests na fila
===

In [95]:
process_request_queue()
con.sql("""
    SELECT 
        request_id, 
        endpoint
    FROM dev.bronze.request_queue WHERE status = 'pending'
    ORDER BY created_at 
""").pl()

10 requests atualizados!


request_id,endpoint
str,str
